In [1]:
# ==========================================================
# AI VOICE ASSISTANT (WEEK 2)
# ==========================================================
#
# Pipeline:
#
# User Speech
#      ↓
# Silero VAD
#      ↓
# Faster-Whisper
#      ↓
# Gemini
#      ↓
# Piper TTS
#      ↓
# Spoken Response
#
# Features:
# - Continuous microphone listening
# - Automatic speech segmentation
# - Low-latency transcription
# - AI-generated responses
# - Speech synthesis
#
# ==========================================================

In [2]:

# ==========================================================
# CELL 2: IMPORT REQUIRED LIBRARIES
# ==========================================================
#
# Libraries used for:
# - Audio capture
# - Speech detection
# - Speech recognition
# - Gemini interaction
# - Text-to-speech
#
# ==========================================================
import os
import re
import time
import queue
import subprocess

import torch
import sounddevice as sd
import numpy as np
import soundfile as sf

from collections import deque
from scipy.io.wavfile import write

from faster_whisper import WhisperModel

from silero_vad import (
    load_silero_vad,
    VADIterator
)

import google.generativeai as genai

C:\Users\Prathamesh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Prathamesh\AppData\Local\Temp\ipykernel_27988\3148683102.py:34: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [3]:
# ==========================================================
# CELL 3: SYSTEM CONFIGURATION AND MODEL LOADING
# ==========================================================
#
# Initializes:
# - Gemini
# - Faster-Whisper Small
# - Silero VAD
#
# Audio Settings:
# - Sample Rate = 16000 Hz
# - Block Size = 512
#
# ==========================================================

GEMINI_API_KEY = "AQ.Ab8RN6LOMk8NrBqxfUBuCRdnZuLvtPf0WPl3qu7pprYchKuBCQ"

genai.configure(
    api_key=GEMINI_API_KEY
)

gemini_model = genai.GenerativeModel(
    "gemini-2.5-flash"
)

# Audio Configuration

SAMPLE_RATE = 16000
BLOCK_SIZE = 512

PRE_SPEECH_SECONDS = 0.5

print("Loading Faster Whisper...")

asr_model = WhisperModel(
    "small",
    device="cuda",
    compute_type="float16"
)

print("Faster Whisper Loaded")

print("Loading Silero VAD...")

vad_model = load_silero_vad()

vad_iterator = VADIterator(
    vad_model,
    sampling_rate=SAMPLE_RATE,
    min_silence_duration_ms=800
)

print("Silero VAD Loaded")

Loading Faster Whisper...
Faster Whisper Loaded
Loading Silero VAD...
Silero VAD Loaded


In [4]:
# ==========================================================
# CELL 4: CONTINUOUS MICROPHONE CAPTURE
# ==========================================================
#
# Creates:
# - Audio Queue
# - Audio Callback
# - Input Stream
#
# Audio frames are continuously pushed into a queue
# for VAD processing.
#
# ==========================================================
audio_queue = queue.Queue()

def audio_callback(
    indata,
    frames,
    time_info,
    status
):

    if status:
        if getattr(status, "input_overflow", False):
            print("Warning: audio input overflow")
        else:
            print(status)

    audio_queue.put(
        indata.copy()
    )

pre_buffer = deque(
    maxlen=int(
        (SAMPLE_RATE * PRE_SPEECH_SECONDS)
        / BLOCK_SIZE
    )
)

print("Microphone Ready")


Microphone Ready


In [5]:
# ==========================================================
# CELL 5: SPEECH SEGMENTATION AND TRANSCRIPTION
# ==========================================================
#
# Functions:
#
# 1. collect_speech_chunk()
#    - Detect speech start
#    - Collect speech audio
#    - Detect speech end
#
# 2. transcribe_chunk()
#    - Convert speech to text
#    - Measure ASR latency
#
# ==========================================================
def collect_speech_chunk():

    global pre_buffer

    collected_audio = []

    speech_active = False

    # Clear any stale queued audio before starting a new capture.
    try:
        while True:
            audio_queue.get_nowait()
    except queue.Empty:
        pass

    print("\nListening...")

    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="float32",
        blocksize=BLOCK_SIZE,
        callback=audio_callback
    ):

        while True:

            chunk = audio_queue.get()

            audio = chunk.flatten()

            pre_buffer.append(audio)

            result = vad_iterator(
                torch.tensor(
                    audio,
                    dtype=torch.float32
                ),
                return_seconds=True
            )

            if result is not None:

                if (
                    "start" in result
                    and not speech_active
                ):

                    speech_active = True

                    collected_audio = list(
                        pre_buffer
                    )

                    print("Speech Started")

                elif (
                    "end" in result
                    and speech_active
                ):

                    print("Speech Ended")

                    speech_active = False

                    audio_chunk = np.concatenate(
                        collected_audio
                    )

                    vad_iterator.reset_states()

                    pre_buffer.clear()

                    return audio_chunk

            if speech_active:

                collected_audio.append(
                    audio
                )


def transcribe_chunk(audio_chunk):

    write(
        "temp_chunk.wav",
        SAMPLE_RATE,
        (
            audio_chunk * 32767
        ).astype(np.int16)
    )

    start_time = time.time()

    segments, _ = asr_model.transcribe(
        "temp_chunk.wav",
        language="en",
        beam_size=5
    )

    transcript = " ".join(
        segment.text
        for segment in segments
    ).strip()

    latency = (
        time.time() - start_time
    )

    return transcript, latency

In [ ]:

# ==========================================================
# CELL 6: RESPONSE GENERATION USING GEMINI
# ==========================================================
#
# Converts user transcripts into concise
# conversational responses.
#
# Response Rules:
# - Maximum 3 sentences
# - Natural speech style
# - No markdown
#
# ==========================================================

def ask_gemini(transcript):

    prompt = f"""
    You are a voice assistant.

    Rules:
    - Keep responses concise
    - Maximum 3 sentences
    - Natural conversational style

    User:
    {transcript}
    """

    try:

        response = gemini_model.generate_content(
            prompt
        )

        return response.text

    except Exception as e:

        print(
            f"Gemini Error: {e}"
        )

        return (
            "Sorry, I am unable to generate "
            "a response right now."
        )


In [ ]:
# ==========================================================
# CELL 7: RESPONSE CLEANING
# ==========================================================
#
# Removes:
# - Markdown symbols
# - Formatting characters
# - Extra whitespace
#
# Produces text suitable for speech synthesis.
#
# ==========================================================
def clean_text(text):

    text = re.sub(
        r'[*_`#>-]+',
        ' ',
        text
    )

    text = re.sub(
        r'\[(.*?)\]\((.*?)\)',
        r'\1',
        text
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    return text

In [ ]:
# ==========================================================
# CELL 8: RESPONSE LOGGING
# ==========================================================
#
# Stores the latest Gemini response in a text file.
#
# This file is later used as input for
# Piper Text-to-Speech synthesis.
#
# ==========================================================
def save_response(text):

    with open(
        "response.txt",
        "w",
        encoding="utf-8"
    ) as f:

        f.write(text)

In [ ]:
# ==========================================================
# CELL 9: TEXT-TO-SPEECH SYNTHESIS
# ==========================================================
#
# Converts Gemini response into speech
# using Piper TTS.
#
# Workflow:
# Response Text
#      ↓
# Piper
#      ↓
# reply.wav
#      ↓
# Playback
#
# ==========================================================
def speak():

    try:

        with open(
            "response.txt",
            "r",
            encoding="utf-8"
        ) as stdin:

            subprocess.run(
                [
                    "piper/piper.exe",
                    "--model",
                    "piper/en_US-amy-medium.onnx",
                    "--config",
                    "piper/en_US-amy-medium.onnx.json",
                    "--output_file",
                    "reply.wav"
                ],
                stdin=stdin,
                check=True
            )

        data, samplerate = sf.read(
            "reply.wav"
        )

        print("\nPlaying Response...\n")

        sd.play(
            data,
            samplerate
        )

        sd.wait()

        print("Playback Finished")

    except Exception as e:

        print(
            f"\nTTS Error: {e}"
        )

In [10]:
# ==========================================================
# CELL 10: MAIN VOICE ASSISTANT LOOP
# ==========================================================
#
# Complete Workflow:
#
# Listen
#   ↓
# VAD
#   ↓
# Faster-Whisper
#   ↓
# Gemini
#   ↓
# Piper
#   ↓
# Audio Response
#
# Special Commands:
# - stop
# - exit
# - quit
# - goodbye
#
# ==========================================================
print("\nVoice Assistant Started")

while True:

    print("\n=== New Interaction ===", flush=True)
    print("Listening for next speech...", flush=True)
    audio_chunk = collect_speech_chunk()

    duration = (
        len(audio_chunk)
        / SAMPLE_RATE
    )

    if duration < 0.5:
        print("Detected too-short audio chunk, listening again...")
        continue

    chunk_end_time = time.time()

    transcript, asr_latency = (
        transcribe_chunk(
            audio_chunk
        )
    )

    if not transcript.strip():
        continue

    total_latency = (
        time.time()
        - chunk_end_time
    )

    print("\n" + "=" * 60, flush=True)

    print("USER:", flush=True)
    print(transcript, flush=True)

    normalized_transcript = re.sub(
        r'[^a-z0-9 ]+',
        ' ',
        transcript.lower()
    ).strip()

    stop_phrases = [
        "stop",
        "exit",
        "quit",
        "goodbye",
        "stop assistant",
        "please stop",
        "stop please"
    ]

    if any(
        normalized_transcript == phrase
        or normalized_transcript.startswith(phrase + " ")
        or normalized_transcript.endswith(" " + phrase)
        or (" " + phrase + " ") in (" " + normalized_transcript + " ")
        for phrase in stop_phrases
    ):
        print("\nAssistant stopped...", flush=True)
        break

    print(
        f"\nASR Latency: "
        f"{total_latency:.2f} sec",
        flush=True
    )

    print("=" * 60, flush=True)

    print(
        "\nGenerating Gemini Response...",
        flush=True
    )

    assistant_response = (
        ask_gemini(
            transcript
        )
    )

    assistant_response = (
        clean_text(
            assistant_response
        )
    )

    print("\nASSISTANT:", flush=True)
    print(
        assistant_response,
        flush=True
    )

    print(
        f"\nResponse Length: "
        f"{len(assistant_response.split())} words",
        flush=True
    )

    save_response(
        assistant_response
    )

    print(
        "\nGenerating Speech...",
        flush=True
    )

    speak()

    time.sleep(2)


Voice Assistant Started

=== New Interaction ===
Listening for next speech...

Listening...
Speech Started
Speech Ended

USER:
What is the difference between machine learning and deep learning?

ASR Latency: 1.21 sec

Generating Gemini Response...

ASSISTANT:
Deep learning is a specialized subfield that falls within the broader discipline of machine learning. While machine learning encompasses a wide range of algorithms, deep learning specifically utilizes multi layered neural networks. These networks are capable of automatically learning complex patterns and representations directly from data, often excelling with very large datasets.

Response Length: 52 words

Generating Speech...

Playing Response...

Playback Finished

=== New Interaction ===
Listening for next speech...

Listening...
Speech Started
Speech Ended

USER:
Stop.

Assistant stopped...
